# CoMusion-GAN — corrected Kaggle notebook

This notebook runs **A0 clean training**, the necessary baseline before evaluating the combined GAN + Transformer + Diffusion architecture. The architecture files and split are also generated. It does **not** report leaked pretrained-pilot scores as validation results.

Attach your prepared Human3.6M dataset as a Kaggle Input, enable a GPU and Internet, then choose **Save Version → Save & Run All**. A run trains up to 20 epochs or 3 hours and saves a checkpoint and log in `/kaggle/working/CoMusionGAN/artifacts`. To continue, attach the previous committed notebook output as an Input and Save & Run All again. The final S9/S11 test remains untouched.

The pretrained official checkpoint saw all 150 training clips; that includes this notebook's 15 held-out validation clips. Earlier A2/A4 12-step pilot metrics were engineering smoke tests only. Train A0, then A2 and A4 from scratch using the same 135/15 split for a valid comparison.


In [ ]:
!nvidia-smi
!pip -q install einops pyyaml "zarr<3"

from pathlib import Path
import os
import json
import csv
import shutil
import zipfile
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable GPU in Kaggle: Settings → Accelerator → GPU."

In [ ]:
WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

OFFICIAL_REPO = WORK / "CoMusion"
DATA_EXTRACT = WORK / "h36m_extracted"

if not OFFICIAL_REPO.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/jsun57/CoMusion.git", str(OFFICIAL_REPO)],
        check=True
    )

# Locate an already-extracted data file, or extract the uploaded ZIP automatically.
data_files = list(INPUT.rglob("data_3d_h36m.npz"))

if not data_files:
    DATA_EXTRACT.mkdir(exist_ok=True)
    archives = [p for p in INPUT.rglob("*") if p.is_file() and zipfile.is_zipfile(p)]

    matching_archives = []
    for archive in archives:
        with zipfile.ZipFile(archive) as zf:
            if any(name.endswith("data_3d_h36m.npz") for name in zf.namelist()):
                matching_archives.append(archive)

    assert matching_archives, (
        "Could not find data_3d_h36m.npz or a ZIP containing it in /kaggle/input. "
        "Attach your prepared Human3.6M dataset."
    )

    print("Extracting:", matching_archives[0].name)
    with zipfile.ZipFile(matching_archives[0]) as zf:
        zf.extractall(DATA_EXTRACT)

    data_files = list(DATA_EXTRACT.rglob("data_3d_h36m.npz"))

assert data_files, "Dataset extraction failed."
source_data_dir = data_files[0].parent

shutil.copytree(source_data_dir, OFFICIAL_REPO / "data", dirs_exist_ok=True)

print("Official source:", OFFICIAL_REPO)
print("Dataset source:", source_data_dir)
print("Prepared data copied to:", OFFICIAL_REPO / "data")

assert (OFFICIAL_REPO / "data" / "data_3d_h36m.npz").exists()
assert (OFFICIAL_REPO / "data" / "data_3d_h36m_test.npz").exists()
print("✅ Dataset preparation passed")

In [ ]:
PROJECT = WORK / "CoMusionGAN"
if not PROJECT.exists():
    shutil.copytree(OFFICIAL_REPO, PROJECT)

import ast
import importlib
import sys

metrics_file = PROJECT / "utils" / "metrics.py"
# Always start from the untouched official copy. This also repairs projects
# previously damaged by the old APD-to-APDE replacement.
metrics_source = (OFFICIAL_REPO / "utils" / "metrics.py").read_text()
metrics_tree = ast.parse(metrics_source)
apd_nodes = [n for n in metrics_tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == "APD"]
assert len(apd_nodes) == 1, "Expected one APD function."
apd_node = apd_nodes[0]

fixed_apd = 'def APD(pred, target, *args, t0=0, t=-1):\n    pred = time_slice(pred, t0, t, 2)\n    batch_size, n_samples = pred.shape[:2]\n    if n_samples < 2:\n        return torch.zeros(batch_size, device=pred.device, dtype=pred.dtype)\n    trajectories = pred.reshape(batch_size, n_samples, -1)\n    distances = torch.cdist(trajectories, trajectories)\n    upper = torch.triu_indices(n_samples, n_samples, offset=1, device=pred.device)\n    return distances[:, upper[0], upper[1]].mean(dim=-1)\n'
lines = metrics_source.splitlines(keepends=True)
apd_start_line = min([apd_node.lineno] + [dec.lineno for dec in apd_node.decorator_list])
metrics_source = "".join(lines[:apd_start_line - 1]) + fixed_apd + "\n" + "".join(lines[apd_node.end_lineno:])
compile(metrics_source, str(metrics_file), "exec")
assert {"ADE", "FDE", "APD", "APDE", "MMADE", "MMFDE"}.issubset({n.name for n in ast.parse(metrics_source).body if isinstance(n, ast.FunctionDef)})
metrics_file.write_text(metrics_source)

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import utils.metrics as metric_module
importlib.reload(metric_module)
from utils.metrics import ADE, FDE, APD
print("✅ APD fixed; ADE/FDE and other metrics preserved")
print("Project:", PROJECT)


In [ ]:
%cd /kaggle/working/CoMusionGAN

from data_utils.dataset_h36m import DatasetH36M

SEED = 2026
T_HIS = 25
T_PRED = 100
rng = np.random.default_rng(SEED)

base_dataset = DatasetH36M(
    mode="train",
    t_his=T_HIS,
    t_pred=T_PRED,
    actions="all",
    augmentation=0,
    stride=1,
    transform=None,
)

clips_by_action = defaultdict(list)

for clip_idx, (subject, action) in base_dataset.dict_indices_inv.items():
    action_base = action.split(" ")[0]
    clips_by_action[action_base].append(
        {
            "clip_idx": int(clip_idx),
            "subject": str(subject),
            "action": str(action),
        }
    )

val_clips = [
    candidates[int(rng.integers(len(candidates)))]
    for _, candidates in sorted(clips_by_action.items())
]
val_clip_ids = {item["clip_idx"] for item in val_clips}

all_clips = [
    {
        "clip_idx": int(clip_idx),
        "subject": str(subject),
        "action": str(action),
    }
    for clip_idx, (subject, action) in base_dataset.dict_indices_inv.items()
]
train_clips = [item for item in all_clips if item["clip_idx"] not in val_clip_ids]

assert len(all_clips) == 150
assert len(train_clips) == 135
assert len(val_clips) == 15
assert {x["clip_idx"] for x in train_clips}.isdisjoint(val_clip_ids)

splits_dir = PROJECT / "splits"
splits_dir.mkdir(exist_ok=True)

split_json = splits_dir / "h36m_train_val_split_seed2026.json"
split_json.write_text(json.dumps({
    "seed": SEED,
    "protocol": {
        "history_frames": T_HIS,
        "future_frames": T_PRED,
        "train_subjects": ["S1", "S5", "S6", "S7", "S8"],
        "final_test_subjects": ["S9", "S11"],
        "split_method": "one complete motion clip per action is held out for validation"
    },
    "train_clips": train_clips,
    "val_clips": val_clips
}, indent=2))

# Convert complete-clip split into individual valid trajectory windows.
train_rows, val_rows = [], []

for clip_idx, start_frame, _ in base_dataset.segments:
    subject, action = base_dataset.dict_indices_inv[int(clip_idx)]
    row = {
        "subject": subject,
        "action": action,
        "pred_init": int(start_frame) + T_HIS,
    }
    (val_rows if int(clip_idx) in val_clip_ids else train_rows).append(row)

train_csv = splits_dir / "h36m_train_segments_seed2026.csv"
val_csv = splits_dir / "h36m_val_segments_seed2026.csv"

pd.DataFrame(train_rows).to_csv(train_csv, index=False)
pd.DataFrame(val_rows).to_csv(val_csv, index=False)

assert len(train_rows) > 0 and len(val_rows) > 0

print(f"Train clips/windows: {len(train_clips)} / {len(train_rows)}")
print(f"Validation clips/windows: {len(val_clips)} / {len(val_rows)}")
print("✅ Fixed split and segment files created")

In [ ]:
dataset_file = PROJECT / "data_utils" / "dataset_h36m.py"
dataset_source = dataset_file.read_text()

old_segment_line = (
    "self.segments_path = None if mode == 'train' else "
    "os.path.join(self.stats_folder, 'segments_test.csv')"
)

new_segment_line = (
    "self.segments_path = segments_path if segments_path is not None else "
    "(None if mode == 'train' else os.path.join(self.stats_folder, 'segments_test.csv'))"
)

assert old_segment_line in dataset_source or new_segment_line in dataset_source, "Unexpected DatasetH36M version."
dataset_source = dataset_source.replace(old_segment_line, new_segment_line, 1)

old_hash_line = (
    "base = [str(self.t_his), str(self.t_pred), str(self.stride), str(self.augmentation)]"
)

new_hash_line = (
    "base = [str(self.t_his), str(self.t_pred), str(self.stride), str(self.augmentation), "
    "os.path.basename(self.segments_path) if self.segments_path else 'all_clips']"
)

assert old_hash_line in dataset_source or new_hash_line in dataset_source, "Could not update statistics identifier safely."
dataset_source = dataset_source.replace(old_hash_line, new_hash_line, 1)

dataset_file.write_text(dataset_source)

# Reload the updated loader and test both partitions.
import importlib
import data_utils.dataset_h36m as h36m_module
importlib.reload(h36m_module)

TrainDataset = h36m_module.DatasetH36M

train_dataset = TrainDataset(
    "train", T_HIS, T_PRED,
    segments_path=str(train_csv),
    augmentation=0, stride=1, transform=None
)

val_dataset = TrainDataset(
    "train", T_HIS, T_PRED,
    segments_path=str(val_csv),
    augmentation=0, stride=1, transform=None
)

train_clip_ids = {int(row[0]) for row in train_dataset.segments}
val_clip_ids_check = {int(row[0]) for row in val_dataset.segments}

assert train_clip_ids.isdisjoint(val_clip_ids_check)
assert train_clip_ids == {x["clip_idx"] for x in train_clips}
assert val_clip_ids_check == val_clip_ids

sample, extra = train_dataset[0]
assert sample.shape == (125, 17, 3)

print("Train windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
print("Sample shape:", sample.shape)
print("✅ Train/validation loader separation passed")

In [ ]:
%cd /kaggle/working/CoMusionGAN

from pathlib import Path
import shutil
import re

diffusion_file = Path("models/GaussianDiffusion.py")
backup_file = Path("models/GaussianDiffusion_original.py")

# The earlier failed cell created this backup before attempting any edits.
if not backup_file.exists():
    shutil.copy2(diffusion_file, backup_file)

source = backup_file.read_text()

# Add loss weights to the diffusion model.
weight_pattern = r"(?m)^        self\.pre_weight = future_weight\s*$"
weight_replacement = """        self.pre_weight = future_weight
        self.lambda_bone = float(getattr(cfg, "lambda_bone", 1.0))
        self.lambda_velocity = float(getattr(cfg, "lambda_velocity", 0.5))
        self.lambda_acceleration = float(getattr(cfg, "lambda_acceleration", 0.5))"""

source, count = re.subn(
    weight_pattern, weight_replacement, source, count=1
)
assert count == 1, "Could not locate the diffusion-loss weights."

# Keep CoMusion's forward pass and history loss. Replace only the future-loss
# portion of p_losses; this does not depend on matching an entire text block.
method_start = source.index("    def p_losses(")
tail_start = source.index("        # 2. future construction:", method_start)
method_end = source.index("    def forward(", tail_start)

new_tail = '''        # Select one complete generated trajectory per input using position loss.
        # All GT-dependent losses below use that same selected trajectory.
        future_errors = loss[..., -t_pred:, :]
        per_sample_position_error = future_errors.mean(dim=(2, 3))
        best_index = per_sample_position_error.argmin(dim=1)
        pre_loss = per_sample_position_error.gather(
            1, best_index[:, None]
        ).mean()

        # CoMusion can return either one output stage [B,K,T,48] or
        # multiple stages [B,K,S,T,48]. Use the final stage for these losses.
        if model_out.ndim == 5:
            pred_future_all = model_out[:, :, -1, -t_pred:, :]
        else:
            pred_future_all = model_out[:, :, -t_pred:, :]

        batch_size = pred_future_all.shape[0]
        row_index = torch.arange(batch_size, device=pred_future_all.device)
        pred_future_best = pred_future_all[row_index, best_index]
        gt_future = target[:, 0, -t_pred:, :]

        # Restore the implicit root joint; compare all k to observed-history bone lengths.
        pred_joints = pred_future_all.reshape(
            batch_size, div_k, t_pred, 16, 3
        )
        # history_bone_reference_v1:
        # Use only the observed history as the bone-length reference.
        pred_root = torch.zeros_like(pred_joints[:, :, :, :1])
        pred_joints = torch.cat((pred_root, pred_joints), dim=3)

        history_joints = raw_x_0_history[:, 0].reshape(
            batch_size, -1, 16, 3
        )
        history_root = torch.zeros_like(history_joints[:, :, :1])
        history_joints = torch.cat(
            (history_root, history_joints), dim=2
        )

        # Parent-child indices for the 17 retained H36M joints.
        parent = [0, 1, 2, 0, 4, 5, 0, 7, 8, 9, 8, 11, 12, 8, 14, 15]
        child  = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

        pred_bones = (
            pred_joints[:, :, :, child]
            - pred_joints[:, :, :, parent]
        )
        history_bones = (
            history_joints[:, :, child]
            - history_joints[:, :, parent]
        )

        pred_lengths = torch.linalg.vector_norm(pred_bones, dim=-1)
        history_lengths = torch.linalg.vector_norm(
            history_bones, dim=-1
        )

        reference_lengths = history_lengths.median(dim=1).values
        reference_lengths = reference_lengths[:, None, None, :].clamp_min(1e-4)

        # Squared relative bone-length error, averaged over all k samples.
        bone_loss = (
            (pred_lengths - reference_lengths) / reference_lengths
        ).square().mean()

        # Include the history/future boundary when measuring motion.
        history_end = raw_x_0_history[:, 0, -2:, :]
        pred_sequence = torch.cat((history_end, pred_future_best), dim=1)
        gt_sequence = torch.cat((history_end, gt_future), dim=1)

        pred_velocity = pred_sequence[:, 1:] - pred_sequence[:, :-1]
        gt_velocity = gt_sequence[:, 1:] - gt_sequence[:, :-1]
        velocity_loss = (pred_velocity - gt_velocity).abs().mean()

        pred_acceleration = pred_velocity[:, 1:] - pred_velocity[:, :-1]
        gt_acceleration = gt_velocity[:, 1:] - gt_velocity[:, :-1]
        acceleration_loss = (
            pred_acceleration - gt_acceleration
        ).abs().mean()

        total_loss = (
            self.his_weight * his_loss
            + self.pre_weight * pre_loss
            + self.lambda_bone * bone_loss
            + self.lambda_velocity * velocity_loss
            + self.lambda_acceleration * acceleration_loss
        )

        extra = {
            "his_loss": self.his_weight * his_loss.item(),
            "pre_loss": self.pre_weight * pre_loss.item(),
            "bone_loss": self.lambda_bone * bone_loss.item(),
            "velocity_loss": self.lambda_velocity * velocity_loss.item(),
            "acceleration_loss": (
                self.lambda_acceleration * acceleration_loss.item()
            ),
        }
        return total_loss, extra

'''

source = source[:tail_start] + new_tail + source[method_end:]

# Check Python syntax before changing the project file.
compile(source, str(diffusion_file), "exec")
diffusion_file.write_text(source)

print("✅ Kinematic losses installed")
print("✅ Original backed up at:", backup_file.resolve())
print("✅ Python syntax check passed")

In [ ]:
from pathlib import Path

discriminator_file = Path("models/temporal_discriminator.py")

discriminator_code = '''import torch
import torch.nn as nn
from torch.nn.utils import spectral_norm


class TemporalDiscriminator(nn.Module):
    """
    Scores a complete 125-frame motion:
    25 observed history frames + 100 future frames.

    At each frame it sees 48 joint positions and 48 velocities.
    """

    def __init__(self):
        super().__init__()

        widths = [96, 128, 256, 256, 256]
        layers = []

        for in_channels, out_channels in zip(widths[:-1], widths[1:]):
            layers.extend([
                spectral_norm(
                    nn.Conv1d(
                        in_channels,
                        out_channels,
                        kernel_size=5,
                        stride=2,
                        padding=2,
                    )
                ),
                nn.LeakyReLU(0.2, inplace=True),
            ])

        self.features = nn.Sequential(*layers)
        self.score = spectral_norm(nn.Linear(256, 1))

    def forward(self, history, future):
        if history.ndim != 3 or future.ndim != 3:
            raise ValueError("Expected [batch, time, 48] tensors")
        if history.shape[1:] != (25, 48):
            raise ValueError(f"Bad history shape: {history.shape}")
        if future.shape[1:] != (100, 48):
            raise ValueError(f"Bad future shape: {future.shape}")
        if history.shape[0] != future.shape[0]:
            raise ValueError("History and future batch sizes differ")

        motion = torch.cat((history, future), dim=1)
        velocity = torch.diff(
            motion,
            dim=1,
            prepend=motion[:, :1],
        )

        features = torch.cat((motion, velocity), dim=-1)
        features = features.transpose(1, 2)  # [B, 96, 125]

        encoded = self.features(features)
        pooled = encoded.mean(dim=-1)
        return self.score(pooled).squeeze(-1)
'''

compile(discriminator_code, str(discriminator_file), "exec")
discriminator_file.write_text(discriminator_code)

print("✅ Discriminator written to:", discriminator_file.resolve())

In [ ]:
ARTIFACTS = PROJECT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
manifest = {
    "project": "CoMusion-GAN",
    "stage": "A0_clean_training",
    "seed": SEED,
    "train_clips": len(train_clips),
    "validation_clips": len(val_clips),
    "final_test_subjects": ["S9", "S11"],
    "validation_warning": "The official pretrained checkpoint saw all 150 training clips, including the 15 held-out here. Its A2/A4 pilot metrics cannot select models.",
    "architecture": "CoMusion transformer-diffusion generator plus history-conditioned temporal GAN discriminator, with bone/velocity/acceleration losses",
    "status": "A0 original diffusion model is being trained from random initialization; A2/A4 full training is a later stage",
}
(ARTIFACTS / "project_manifest.json").write_text(json.dumps(manifest, indent=2))
for path in (split_json, train_csv, val_csv):
    shutil.copy2(path, ARTIFACTS / path.name)
shutil.copy2(PROJECT / "models" / "GaussianDiffusion.py", ARTIFACTS / "GaussianDiffusion_with_kinematic_losses.py")
shutil.copy2(PROJECT / "models" / "temporal_discriminator.py", ARTIFACTS / "temporal_discriminator.py")
print("✅ Split and architecture artifacts staged")


In [ ]:
%cd /kaggle/working/CoMusionGAN

import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from utils import Config
from models.load_models import get_model
from models.GaussianDiffusion import GaussianDiffusion
from models.temporal_discriminator import TemporalDiscriminator
from data_utils.dataset_h36m import (
    DatasetH36M,
    generate_h36_loss_weights,
)
from data_utils.transforms import DataAugmentation
from utils.metrics import ADE, FDE, APD

PROJECT = Path("/kaggle/working/CoMusionGAN")
ARTIFACTS = PROJECT / "artifacts"
SPLITS = PROJECT / "splits"
DEVICE = torch.device("cuda")

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings."

cfg = Config("h36m", test=True)
assert (cfg.t_his, cfg.t_pred, cfg.node_n, cfg.diffuse_steps, cfg.div_k) == (
    25, 100, 48, 10, 2
)

split = json.loads(
    (SPLITS / "h36m_train_val_split_seed2026.json").read_text()
)
expected_train = {x["clip_idx"] for x in split["train_clips"]}
expected_val = {x["clip_idx"] for x in split["val_clips"]}

train_ds = DatasetH36M(
    "train", 25, 100,
    segments_path=str(SPLITS / "h36m_train_segments_seed2026.csv"),
    augmentation=0,
    stride=1,
    transform=transforms.Compose([DataAugmentation(cfg.rota_prob)]),
)
train_ds.stride = cfg.stride
train_ds.augmentation = cfg.augmentation

val_ds = DatasetH36M(
    "train", 25, 100,
    segments_path=str(SPLITS / "h36m_val_segments_seed2026.csv"),
    augmentation=0,
    stride=1,
    transform=None,
)

actual_train = {int(row[0]) for row in train_ds.segments}
actual_val = {int(row[0]) for row in val_ds.segments}

assert actual_train == expected_train
assert actual_val == expected_val
assert actual_train.isdisjoint(actual_val)

windows_by_clip = defaultdict(list)
for index, row in enumerate(val_ds.segments):
    windows_by_clip[int(row[0])].append(index)

pilot_indices = [
    windows[len(windows) // 2]
    for _, windows in sorted(windows_by_clip.items())
]
assert len(pilot_indices) == 15
pilot_val_ds = Subset(val_ds, pilot_indices)

print("✅ Clean A0 setup ready; no official checkpoint is loaded")


In [ ]:
def to_motion(batch):
    # Dataset contains 17 joints including root; the model predicts 16.
    return batch[..., 1:, :].reshape(
        batch.shape[0], 125, 48
    ).to(DEVICE, dtype=torch.float32)

def mean_relative_bone_stretch(history, predictions):
    """Mean relative bone-length error across every predicted sample."""
    batch, samples = predictions.shape[:2]

    hist_joints = history.reshape(batch, 25, 16, 3)
    pred_joints = predictions.reshape(batch, samples, 100, 16, 3)

    hist_joints = torch.cat(
        (torch.zeros_like(hist_joints[:, :, :1]), hist_joints), dim=2
    )
    pred_joints = torch.cat(
        (torch.zeros_like(pred_joints[:, :, :, :1]), pred_joints), dim=3
    )

    parent = [0, 1, 2, 0, 4, 5, 0, 7, 8, 9, 8, 11, 12, 8, 14, 15]
    child  = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

    hist_lengths = torch.linalg.vector_norm(
        hist_joints[:, :, child] - hist_joints[:, :, parent],
        dim=-1,
    )
    pred_lengths = torch.linalg.vector_norm(
        pred_joints[:, :, :, child] - pred_joints[:, :, :, parent],
        dim=-1,
    )

    reference = hist_lengths.median(dim=1).values
    reference = reference[:, None, None, :].clamp_min(1e-4)
    return ((pred_lengths - reference) / reference).abs().mean()

@torch.no_grad()
def pilot_validation(diffusion, sample_count=8):
    diffusion.eval()
    loader = DataLoader(
        pilot_val_ds, batch_size=1, shuffle=False, num_workers=0
    )

    records = []
    for batch, _ in loader:
        motion = to_motion(batch)
        history = motion[:, :25]
        target = motion[:, 25:]

        repeated_history = history.repeat_interleave(
            sample_count, dim=0
        )
        predictions = diffusion.sample(
            repeated_history,
            batch_size=sample_count,
            uncond=True,
            clip_denoised=False,
        ).reshape(1, sample_count, 100, 48)

        records.append({
            "ADE": ADE(predictions, target).item(),
            "FDE": FDE(predictions, target).item(),
            "APD": APD(predictions, target).item(),
            "all_sample_stretch": mean_relative_bone_stretch(
                history, predictions
            ).item(),
        })

    return {
        key: float(np.mean([row[key] for row in records]))
        for key in records[0]
    }

print('✅ Motion conversion and validation helpers ready')


In [ ]:
%cd /kaggle/working/CoMusionGAN

import hashlib
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from models.load_models import get_model
from models.GaussianDiffusion_original import (
    GaussianDiffusion as OriginalGaussianDiffusion
)
from data_utils.dataset_h36m import generate_h36_loss_weights
from utils.util import get_scheduler

assert len(actual_train) == 135
assert len(actual_val) == 15
assert actual_train.isdisjoint(actual_val)

A0_TOTAL_EPOCHS = 500
A0_EPOCHS_THIS_RUN = 20
A0_MAX_HOURS = 3
MICRO_BATCH = 8
ACCUMULATE = 8  # effective batch size = 64

split_bytes = (
    SPLITS / "h36m_train_val_split_seed2026.json"
).read_bytes()
split_hash = hashlib.sha256(split_bytes).hexdigest()

history_weights = generate_h36_loss_weights(
    cfg.t_his, scale=cfg.loss_weight_scale
)
future_weights = generate_h36_loss_weights(
    cfg.t_pred, scale=cfg.loss_weight_scale
)
loss_weights = torch.cat(
    (history_weights, future_weights), dim=0
)

# Fresh model initialization: deliberately do not load the official checkpoint.
random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

a0_model = get_model(cfg).to(DEVICE)
a0_diffusion = OriginalGaussianDiffusion(
    model=a0_model,
    cfg=cfg,
    future_motion_size=(cfg.t_pred, cfg.node_n),
    timesteps=cfg.diffuse_steps,
    loss_type=cfg.loss_type,
    objective=cfg.objective,
    beta_schedule=cfg.beta_schedule,
    history_weight=cfg.history_weight,
    future_weight=cfg.future_weight,
    st_loss_weight=loss_weights,
).to(DEVICE)

a0_optimizer = torch.optim.AdamW(
    a0_diffusion.parameters(),
    lr=cfg.train_lr,
    betas=(0.9, 0.99),
    weight_decay=cfg.weight_decay,
)
a0_scheduler = get_scheduler(
    a0_optimizer,
    policy=cfg.sched_policy,
    nepoch_fix=cfg.num_epoch_fix_lr,
    nepoch=cfg.train_epoch,
)
a0_scaler = torch.amp.GradScaler("cuda")

checkpoint_out = ARTIFACTS / "A0_clean_latest.pth"
log_out = ARTIFACTS / "A0_clean_training_log.csv"

# For a later chunk, attach the previous committed notebook output as
# a Kaggle Input. This finds its A0 checkpoint automatically.
resume_candidates = list(
    Path("/kaggle/input").rglob("A0_clean_latest.pth")
)
assert len(resume_candidates) <= 1, (
    "More than one A0 checkpoint is attached. Keep only the latest one."
)

start_epoch = 0
a0_history = []

if resume_candidates:
    resume = torch.load(
        resume_candidates[0],
        map_location=DEVICE,
        weights_only=False,
    )
    assert resume["split_sha256"] == split_hash, (
        "Attached checkpoint used a different validation split."
    )
    assert resume["variant"] == "A0_clean"

    a0_diffusion.load_state_dict(resume["generator"])
    a0_optimizer.load_state_dict(resume["optimizer"])
    a0_scheduler.load_state_dict(resume["scheduler"])
    a0_scaler.load_state_dict(resume["scaler"])
    start_epoch = int(resume["epoch_completed"])
    a0_history = list(resume["history"])
    print(f"Resuming clean A0 from epoch {start_epoch}")
else:
    print("Starting clean A0 from scratch on 135 training clips")

assert start_epoch <= A0_TOTAL_EPOCHS

end_epoch = min(
    A0_TOTAL_EPOCHS,
    start_epoch + A0_EPOCHS_THIS_RUN,
)
run_started = time.monotonic()

def save_a0(epoch_completed):
    payload = {
        "variant": "A0_clean",
        "epoch_completed": epoch_completed,
        "total_epochs_planned": A0_TOTAL_EPOCHS,
        "split_sha256": split_hash,
        "train_clips": 135,
        "validation_clips": 15,
        "generator": a0_diffusion.state_dict(),
        "optimizer": a0_optimizer.state_dict(),
        "scheduler": a0_scheduler.state_dict(),
        "scaler": a0_scaler.state_dict(),
        "history": a0_history,
    }
    temporary = checkpoint_out.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(checkpoint_out)
    pd.DataFrame(a0_history).to_csv(log_out, index=False)

for epoch in range(start_epoch, end_epoch):
    random.seed(2026 + epoch)
    np.random.seed(2026 + epoch)
    torch.manual_seed(2026 + epoch)
    torch.cuda.manual_seed_all(2026 + epoch)

    train_loader = DataLoader(
        train_ds,
        batch_size=MICRO_BATCH,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
        generator=torch.Generator().manual_seed(2026 + epoch),
    )

    # Process complete groups of 8 micro-batches.
    micro_batches = (len(train_loader) // ACCUMULATE) * ACCUMULATE
    assert micro_batches > 0

    a0_diffusion.train()
    a0_optimizer.zero_grad(set_to_none=True)
    losses = []
    epoch_started = time.monotonic()

    for index, (batch, _) in enumerate(train_loader):
        if index >= micro_batches:
            break

        motion = to_motion(batch)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            loss, _ = a0_diffusion(
                motion, None,
                div_k=cfg.div_k,
                uncond=True,
                mmgt=None,
            )

        assert torch.isfinite(loss), (
            f"Non-finite A0 loss in epoch {epoch + 1}"
        )
        a0_scaler.scale(loss / ACCUMULATE).backward()
        losses.append(loss.item())

        if (index + 1) % ACCUMULATE == 0:
            a0_scaler.unscale_(a0_optimizer)
            torch.nn.utils.clip_grad_norm_(
                a0_diffusion.parameters(), max_norm=1.0
            )
            a0_scaler.step(a0_optimizer)
            a0_scaler.update()
            a0_optimizer.zero_grad(set_to_none=True)

    a0_scheduler.step()
    torch.cuda.synchronize()

    record = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(losses)),
        "epoch_seconds": time.monotonic() - epoch_started,
        "learning_rate": a0_optimizer.param_groups[0]["lr"],
        "val_ADE_15x8": float("nan"),
        "val_FDE_15x8": float("nan"),
        "val_APD_15x8": float("nan"),
        "val_all_sample_stretch_15x8": float("nan"),
    }

    if (epoch + 1) % 5 == 0:
        validation = pilot_validation(
            a0_diffusion, sample_count=8
        )
        record["val_ADE_15x8"] = validation["ADE"]
        record["val_FDE_15x8"] = validation["FDE"]
        record["val_APD_15x8"] = validation["APD"]
        record["val_all_sample_stretch_15x8"] = (
            validation["all_sample_stretch"]
        )

    a0_history.append(record)
    save_a0(epoch + 1)

    print(
        f"A0 clean epoch {epoch + 1}/{A0_TOTAL_EPOCHS} "
        f"loss={record['train_loss']:.4f} "
        f"seconds={record['epoch_seconds']:.1f} "
        f"val_ADE={record['val_ADE_15x8']:.4f}",
        flush=True,
    )

    if (time.monotonic() - run_started) / 3600 >= A0_MAX_HOURS:
        print("Reached this run's 3-hour limit; checkpoint saved.")
        break

assert checkpoint_out.exists() and checkpoint_out.stat().st_size > 0
assert log_out.exists() and log_out.stat().st_size > 0

print("\nA0 CLEAN CHUNK SAVED ✅")
print("Completed epoch:", a0_history[-1]["epoch"] if a0_history else start_epoch)
print("Checkpoint:", checkpoint_out)
print("Log:", log_out)

In [ ]:
assert (ARTIFACTS / "A0_clean_latest.pth").exists()
assert (ARTIFACTS / "A0_clean_training_log.csv").exists()
print("\nSAVE & RUN ALL COMPLETE ✅")
print("Latest clean A0 epoch:", a0_history[-1]["epoch"])
print("Training subjects: S1, S5, S6, S7, S8; final test S9/S11 untouched")
print("Saved files:")
for path in sorted(ARTIFACTS.iterdir()):
    print(" -", path.name, round(path.stat().st_size / 1e6, 2), "MB")
print("For the next 20-epoch chunk, attach this committed notebook's output as a Kaggle Input, then Save & Run All again.")
